In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [2]:
yihan = pd.read_csv("../data/processed/clean_sg_part_a_yihan.csv")
junhui = pd.read_csv("../data/processed/clean_sg_part_b_junhui.csv")

print("Yihan shape:", yihan.shape)
print("Junhui shape:", junhui.shape)

Yihan shape: (2012, 5)
Junhui shape: (2012, 7)


In [3]:
sg = pd.merge(
    yihan,
    junhui,
    on="D_INTERVIEW",
    how="inner"
)

print("Merged shape:", sg.shape)
sg.head()

Merged shape: (2012, 11)


,D_INTERVIEW,Q131,Q138,Q260,Q262,Q275,Q288,Q279,Q273,Q144,Q145
0,702070001.0,2.0,4.0,2.0,37.0,7.0,5.0,1.0,1.0,2.0,2.0
1,702070002.0,2.0,4.0,2.0,42.0,6.0,1.0,1.0,6.0,-2.0,1.0
2,702070003.0,2.0,3.0,1.0,28.0,6.0,4.0,1.0,6.0,2.0,2.0
3,702070004.0,2.0,3.0,2.0,30.0,6.0,6.0,1.0,1.0,2.0,2.0
4,702070005.0,1.0,4.0,2.0,32.0,3.0,8.0,5.0,1.0,2.0,2.0


In [4]:
sg = sg.rename(columns={
    "Q131": "security",
    "Q138": "harassment",
    "Q260": "gender",
    "Q262": "age",
    "Q275": "education",
    "Q288": "household_income",
    "Q279": "employment",
    "Q273": "marital_status",
    "Q144": "victim_self_raw",
    "Q145": "victim_family_raw"
})

sg.head()

,D_INTERVIEW,security,harassment,gender,age,education,household_income,employment,marital_status,victim_self_raw,victim_family_raw
0,702070001.0,2.0,4.0,2.0,37.0,7.0,5.0,1.0,1.0,2.0,2.0
1,702070002.0,2.0,4.0,2.0,42.0,6.0,1.0,1.0,6.0,-2.0,1.0
2,702070003.0,2.0,3.0,1.0,28.0,6.0,4.0,1.0,6.0,2.0,2.0
3,702070004.0,2.0,3.0,2.0,30.0,6.0,6.0,1.0,1.0,2.0,2.0
4,702070005.0,1.0,4.0,2.0,32.0,3.0,8.0,5.0,1.0,2.0,2.0


negative missing values

In [5]:
vars_to_clean = [
    "security",
    "harassment",
    "gender",
    "age",
    "education",
    "household_income",
    "employment",
    "marital_status",
    "victim_self_raw",
    "victim_family_raw"
]

sg[vars_to_clean] = sg[vars_to_clean].mask(
    sg[vars_to_clean] < 0
)

Reverse code security

In [6]:
sg["security_r"] = 5 - sg["security"]

Reverse code harassment

In [7]:
sg["harassment_r"] = 5 - sg["harassment"]

Gender → gender_bi

In [8]:
sg["gender_bi"] = sg["gender"].map({
    1: 0,
    2: 1
})

Employment → em

In [9]:
sg["em"] = sg["employment"].map({
    1: 1,
    2: 1,
    3: 1,
    4: 0,
    5: 0,
    6: 0,
    7: 0,
    8: 0
})

1 = employed
0 = not employed

In [10]:
sg["marriage"] = sg["marital_status"].map({
    1: 1,
    2: 1,
    3: 0,
    4: 0,
    5: 0,
    6: 0
})

1 = married / living as married
0 = otherwise

In [11]:
sg["victim_self"] = sg["victim_self_raw"].map({
    1: 1,
    2: 0
})

sg["victim_family"] = sg["victim_family_raw"].map({
    1: 1,
    2: 0
})

1 = victim
0 = not victim

In [12]:
recode_vars = [
    "security_r",
    "harassment_r",
    "gender_bi",
    "em",
    "marriage",
    "victim_self",
    "victim_family"
]

for var in recode_vars:
    print("\n--------------------")
    print(var)
    print(sg[var].value_counts(dropna=False).sort_index())


--------------------
security_r
security_r
1.0       9
2.0     152
3.0    1093
4.0     756
NaN       2
Name: count, dtype: int64

--------------------
harassment_r
harassment_r
1.0    1453
2.0     466
3.0      66
4.0      16
NaN      11
Name: count, dtype: int64

--------------------
gender_bi
gender_bi
0     924
1    1088
Name: count, dtype: int64

--------------------
em
em
0     720
1    1292
Name: count, dtype: int64

--------------------
marriage
marriage
0     803
1    1209
Name: count, dtype: int64

--------------------
victim_self
victim_self
0.0    1932
1.0      79
NaN       1
Name: count, dtype: int64

--------------------
victim_family
victim_family
0.0    1927
1.0      66
NaN      19
Name: count, dtype: int64


Descriptive Statistics

Security frequency + percentage

In [13]:
sg["security_r"].value_counts(
    dropna=False
).sort_index()

security_r
1.0       9
2.0     152
3.0    1093
4.0     756
NaN       2
Name: count, dtype: int64

In [14]:
sg["security_r"].value_counts(
    normalize=True,
    dropna=False
).sort_index() * 100

security_r
1.0     0.447316
2.0     7.554672
3.0    54.324056
4.0    37.574553
NaN     0.099404
Name: proportion, dtype: float64

Harassment frequency + percentage

In [15]:
sg["harassment_r"].value_counts(
    dropna=False
).sort_index()

harassment_r
1.0    1453
2.0     466
3.0      66
4.0      16
NaN      11
Name: count, dtype: int64

In [16]:
sg["harassment_r"].value_counts(
    normalize=True,
    dropna=False
).sort_index() * 100

harassment_r
1.0    72.216700
2.0    23.161034
3.0     3.280318
4.0     0.795229
NaN     0.546720
Name: proportion, dtype: float64

Gender frequency + percentage

In [17]:
sg["gender_bi"].value_counts(
    dropna=False
).sort_index()

gender_bi
0     924
1    1088
Name: count, dtype: int64

In [18]:
sg["gender_bi"].value_counts(
    normalize=True,
    dropna=False
).sort_index() * 100

gender_bi
0    45.924453
1    54.075547
Name: proportion, dtype: float64

0 = Men
1 = Women

Security / Harassment by Gender

In [19]:
sg.groupby("gender_bi")[["security_r", "harassment_r"]].agg(
    ["count", "mean", "std"]
)

security_r                    harassment_r                    
               count     mean       std        count      mean       std
gender_bi                                                               
0                923  3.35428  0.608286          920  1.306522  0.563257
1               1087  3.23827  0.625787         1081  1.336725  0.588022

Security / Harassment / Age summary

In [20]:
summary_stats = pd.DataFrame({
    "Obs": sg[["security_r", "harassment_r", "age"]].count(),
    "mean": sg[["security_r", "harassment_r", "age"]].mean(),
    "median": sg[["security_r", "harassment_r", "age"]].median(),
    "sd": sg[["security_r", "harassment_r", "age"]].std(),
    "min": sg[["security_r", "harassment_r", "age"]].min(),
    "max": sg[["security_r", "harassment_r", "age"]].max()
})

summary_stats.round(3)

,Obs,mean,median,sd,min,max
security_r,2010,3.292,3.0,0.620,1.0,4.0
harassment_r,2001,1.323,1.0,0.577,1.0,4.0
age,2012,47.783,47.0,16.235,21.0,91.0


education
household_income
em
marriage
victim_self
victim_family
gender_bi

In [21]:
def category_table(df, variables):
    tables = []

    for var in variables:
        counts = df[var].value_counts(
            dropna=True
        )

        percents = df[var].value_counts(
            normalize=True,
            dropna=True
        ) * 100

        temp = pd.DataFrame({
            "Item": var,
            "Category": counts.index,
            "Count": counts.values,
            "N": df[var].notna().sum(),
            "Percent": percents.values
        })

        tables.append(temp)

    return pd.concat(
        tables,
        ignore_index=True
    )

In [22]:
categorical_summary = category_table(
    sg,
    [
        "education",
        "household_income",
        "em",
        "marriage",
        "victim_self",
        "victim_family",
        "gender_bi"
    ]
)

categorical_summary["Percent"] = (
    categorical_summary["Percent"].round(2)
)

categorical_summary

,Item,Category,Count,N,Percent
0,education,6.0,502,2008,25.00
1,education,5.0,348,2008,17.33
2,education,3.0,341,2008,16.98
3,education,4.0,272,2008,13.55
4,education,7.0,155,2008,7.72
5,education,2.0,130,2008,6.47
6,education,1.0,126,2008,6.27
7,education,0.0,117,2008,5.83
8,education,8.0,17,2008,0.85
9,household_income,5.0,588,1956,30.06


Correlation Table

In [23]:
corr_vars = [
    "security_r",
    "harassment_r",
    "gender_bi",
    "education",
    "household_income",
    "em",
    "marriage",
    "victim_self",
    "victim_family"
]

Spearman correlation

In [24]:
corr_table = (
    sg[corr_vars]
    .corr(method="spearman")
    .round(3)
)

corr_table

,security_r,harassment_r,gender_bi,education,household_income,em,marriage,victim_self,victim_family
security_r,1.000,-0.189,-0.094,-0.029,0.128,-0.020,0.060,-0.076,-0.106
harassment_r,-0.189,1.000,0.027,0.084,-0.037,0.010,-0.062,0.122,0.140
gender_bi,-0.094,0.027,1.000,-0.039,-0.013,-0.122,-0.028,-0.040,-0.004
education,-0.029,0.084,-0.039,1.000,0.373,0.297,-0.007,-0.003,0.052
household_income,0.128,-0.037,-0.013,0.373,1.000,0.131,0.127,-0.102,-0.025
em,-0.020,0.010,-0.122,0.297,0.131,1.000,0.020,0.002,0.033
marriage,0.060,-0.062,-0.028,-0.007,0.127,0.020,1.000,-0.071,-0.045
victim_self,-0.076,0.122,-0.040,-0.003,-0.102,0.002,-0.071,1.000,0.288
victim_family,-0.106,0.140,-0.004,0.052,-0.025,0.033,-0.045,0.288,1.000


In [25]:
for i in range(len(corr_vars)):
    for j in range(i + 1, len(corr_vars)):

        var1 = corr_vars[i]
        var2 = corr_vars[j]

        temp = sg[[var1, var2]].dropna()

        rho, p = spearmanr(
            temp[var1],
            temp[var2]
        )

        if p < 0.001:
            p_text = "p < .001"
        else:
            p_text = f"p = {p:.3f}"

        print(
            f"{var1} × {var2}: "
            f"rho = {rho:.3f}, {p_text}"
        )

security_r × harassment_r: rho = -0.189, p < .001
security_r × gender_bi: rho = -0.094, p < .001
security_r × education: rho = -0.029, p = 0.201
security_r × household_income: rho = 0.128, p < .001
security_r × em: rho = -0.020, p = 0.360
security_r × marriage: rho = 0.060, p = 0.007
security_r × victim_self: rho = -0.076, p < .001
security_r × victim_family: rho = -0.106, p < .001
harassment_r × gender_bi: rho = 0.027, p = 0.232
harassment_r × education: rho = 0.084, p < .001
harassment_r × household_income: rho = -0.037, p = 0.099
harassment_r × em: rho = 0.010, p = 0.670
harassment_r × marriage: rho = -0.062, p = 0.006
harassment_r × victim_self: rho = 0.122, p < .001
harassment_r × victim_family: rho = 0.140, p < .001
gender_bi × education: rho = -0.039, p = 0.082
gender_bi × household_income: rho = -0.013, p = 0.557
gender_bi × em: rho = -0.122, p < .001
gender_bi × marriage: rho = -0.028, p = 0.209
gender_bi × victim_self: rho = -0.040, p = 0.076
gender_bi × victim_family: rho = 